# 73 - Experimento MPNet + DBSCAN

Este notebook compara embeddings `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` com DBSCAN para agrupar descricoes normalizadas de itens de supermercado. Compara duas sementes de amostragem, 42 e 123, usando a mesma base e periodo do notebook 72. Nao usa NCM nem rotulos humanos; ruido e tratado como resultado explicito.

Execute de cima para baixo em uma sessao limpa. Cada execucao grava relatorios em uma pasta UTC nova dentro de `reports/mpnet_dbscan/`. O primeiro uso do MPNet pode baixar o modelo do Hugging Face.


In [7]:
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook73")
pd.set_option("display.max_colwidth", 140)


## 1. Parametros

`eps` e o raio maximo de vizinhanca em distancia cosseno; `min_samples` e o minimo de pontos para formar uma regiao densa. Silhouette e calculada somente nos itens nao classificados como ruido. Frequencia pondera o ajuste, enquanto cada descricao unica continua sendo um ponto. A grade e as amostras sao exploratorias; resultados devem ser lidos junto da quantidade de clusters e da fracao de ruido.

O experimento repete a grade com as sementes 42 e 123 para avaliar a sensibilidade a amostragem. MPNet gera embeddings com o tokenizer e a representacao definidos pelo modelo; nao ha um parametro `analyzer` equivalente ao do TF-IDF. Para comparar analyzers, use o notebook 72.


In [8]:
DATASET_NAME = "items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto"
PERIOD_LABEL = "202501-202504"
DATASET_RELATIVE_PATH = Path("data/processed/items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto/items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto-202501-202504.parquet")
DATASET_PATH = next(
    (root / DATASET_RELATIVE_PATH for root in (Path.cwd(), *Path.cwd().parents)
     if (root / DATASET_RELATIVE_PATH).is_file()),
    DATASET_RELATIVE_PATH,
)
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Dataset nao encontrado: {DATASET_RELATIVE_PATH}")
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = None  # None usa todas as descricoes.
SAMPLE_SEEDS = [42, 123, 2026]
ENCODE_BATCH_SIZE = 64
EPS_VALUES = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.90]
MIN_SAMPLES_VALUES = [5, 10, 15, 20, 40, 70, 100]
OUTPUT_ROOT = Path("reports/mpnet_dbscan")


## 2. Fun??es da an?lise

A prepara??o consolida duplicatas sem alterar a frequ?ncia total. O resumo mant?m o r?tulo `-1` para ru?do. A silhouette exclui ru?do e s? ? calculada quando h? pelo menos dois clusters e mais de um item em cada parti??o considerada.


In [9]:
def prepare_items(frame):
    """Limpa descri??es, agrega duplicatas e preserva a frequ?ncia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def score_configuration(vectors, labels):
    """Calcula m?tricas de DBSCAN e silhouette sem incluir pontos de ru?do."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_labels = labels[~noise_mask]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    if 2 <= cluster_count < len(clustered_labels):
        score = float(silhouette_score(vectors[~noise_mask], clustered_labels, metric="cosine"))
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_cosine_without_noise": score,
    }


def summarize_clusters(items, labels):
    """Retorna atribui??es e resumo dos clusters, incluindo a linha de ru?do."""
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary


## 3. Carregar, preparar e gerar embeddings

A amostra e selecionada separadamente para cada semente. O modelo MPNet transforma cada amostra em embeddings normalizados, que serao reutilizados em toda a grade de DBSCAN daquela semente.


In [10]:
base_items = prepare_items(pd.read_parquet(DATASET_PATH))
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

items_by_seed = {}
vectors_by_seed = {}
for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    items_by_seed[sample_seed] = sampled_items
    vectors_by_seed[sample_seed] = vectors
    logger.info(
        "Seed %d: %d descricoes, embeddings %s, frequencia total %d",
        sample_seed, len(sampled_items), vectors.shape, sampled_items[FREQUENCY].sum(),
    )

display(pd.DataFrame([
    {
        "sample_seed": sample_seed,
        "descricoes": len(items_by_seed[sample_seed]),
        "dimensoes_embedding": vectors_by_seed[sample_seed].shape[1],
        "frequencia_total": int(items_by_seed[sample_seed][FREQUENCY].sum()),
    }
    for sample_seed in SAMPLE_SEEDS
]))


2026-10-11 10:12:06,917 INFO Descricoes disponiveis apos preparacao: 70863
2026-10-11 10:12:06,917 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-10-11 10:12:07,019 INFO No device provided, using cpu
2026-10-11 10:12:07,498 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-11 10:12:07,536 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-10-11 10:12:07,709 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 30

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-11 10:12:14,550 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-11 10:12:14,849 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-11 10:12:15,049 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-11 10:12:24,349 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-11 10:12:24,570 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-11 10:12:24,606 INFO HTTP Request: HEAD https:

Batches:   0%|          | 0/1108 [00:00<?, ?it/s]

2026-10-11 10:35:07,572 INFO Seed 42: 70863 descricoes, embeddings (70863, 768), frequencia total 285364


Batches:   0%|          | 0/1108 [00:00<?, ?it/s]

2026-10-11 10:57:20,533 INFO Seed 123: 70863 descricoes, embeddings (70863, 768), frequencia total 285364


Batches:   0%|          | 0/1108 [00:00<?, ?it/s]

2026-10-11 11:35:40,640 INFO Seed 2026: 70863 descricoes, embeddings (70863, 768), frequencia total 285364


,sample_seed,descricoes,dimensoes_embedding,frequencia_total
0,42,70863,768,285364
1,123,70863,768,285364
2,2026,70863,768,285364


## 4. Ajustar DBSCAN e comparar parametros

DBSCAN usa distancia cosseno nos vetores densos do MPNet. A frequencia entra como `sample_weight`; silhouette e fracao de ruido contam descricoes unicas. Os embeddings de cada semente sao reutilizados nos ajustes correspondentes.


In [11]:
records = []
labels_by_run = {}
items_by_run = {}
for sample_seed in SAMPLE_SEEDS:
    items = items_by_seed[sample_seed]
    vectors = vectors_by_seed[sample_seed]
    for eps in EPS_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            experiment_id = f"seed{sample_seed}_eps{eps:g}_min_samples{min_samples}"
            logger.info("Ajustando DBSCAN %s", experiment_id)
            model = DBSCAN(
                eps=eps, min_samples=min_samples, metric="cosine",
                algorithm="brute", n_jobs=-1,
            )
            model.fit(vectors, sample_weight=items[FREQUENCY].to_numpy(dtype=float))
            labels = model.labels_
            metrics = score_configuration(vectors, labels)
            records.append({
                "run_id": experiment_id, "sample_seed": sample_seed,
                "eps": eps, "min_samples": min_samples,
                "frequency_weighted": True, **metrics,
            })
            labels_by_run[experiment_id] = labels
            items_by_run[experiment_id] = items
results = pd.DataFrame(records).sort_values(
    ["silhouette_cosine_without_noise", "noise_fraction"],
    ascending=[False, True], na_position="last",
).reset_index(drop=True)
display(results)


2026-10-11 11:35:40,748 INFO Ajustando DBSCAN seed42_eps0.01_min_samples5
2026-10-11 11:36:47,261 INFO Ajustando DBSCAN seed42_eps0.01_min_samples10
2026-10-11 11:37:48,081 INFO Ajustando DBSCAN seed42_eps0.01_min_samples15
2026-10-11 11:38:47,677 INFO Ajustando DBSCAN seed42_eps0.01_min_samples20
2026-10-11 11:39:46,824 INFO Ajustando DBSCAN seed42_eps0.01_min_samples40
2026-10-11 11:40:47,687 INFO Ajustando DBSCAN seed42_eps0.01_min_samples70
2026-10-11 11:41:47,883 INFO Ajustando DBSCAN seed42_eps0.01_min_samples100
2026-10-11 11:42:48,196 INFO Ajustando DBSCAN seed42_eps0.05_min_samples5
2026-10-11 11:43:59,805 INFO Ajustando DBSCAN seed42_eps0.05_min_samples10
2026-10-11 11:45:06,881 INFO Ajustando DBSCAN seed42_eps0.05_min_samples15
2026-10-11 11:46:12,543 INFO Ajustando DBSCAN seed42_eps0.05_min_samples20
2026-10-11 11:47:15,426 INFO Ajustando DBSCAN seed42_eps0.05_min_samples40
2026-10-11 11:48:17,484 INFO Ajustando DBSCAN seed42_eps0.05_min_samples70
2026-10-11 11:49:18,146 IN

MemoryError: bad allocation

## 5. Revisar um resultado e salvar relat?rios

A sele??o autom?tica usa a maior silhouette dispon?vel; confira tamb?m quantos clusters foram formados e a fra??o de ru?do antes de interpretar. Todos os resultados da grade s?o salvos, e os arquivos da execu??o n?o sobrescrevem resultados anteriores.


In [ ]:
SELECTED_RUN_ID = results.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
display(cluster_summary)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "dataset_path": str(DATASET_RELATIVE_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "embedding_normalized": True, "frequency_weighted": True,
    "selected_run_id": SELECTED_RUN_ID, "eps_values": EPS_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
}
(output_dir / "run.json").write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
logger.info("Relatorios salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,17338,49760
5,4,422,1821
4,3,197,1907
3,2,162,1604
61,60,124,129
...,...,...,...
96,95,1,160
105,104,1,107
103,102,1,201
101,100,1,180


2026-10-11 08:02:58,319 INFO Relatorios salvos em reports\mpnet_dbscan\202501-202504_20261011T110258Z


### Interpreta??o

- `noise_fraction` alta indica que muitas descri??es ficaram fora de regi?es densas para os par?metros escolhidos.
- Silhouette deve ser interpretada junto da quantidade de clusters e da fra??o de ru?do; uma pontua??o alta pode refletir poucos pontos classificados.
- A compara??o com TF-IDF ? explorat?ria: os dois m?todos usam a mesma amostra, mas representam similaridade de formas diferentes.
- DBSCAN n?o fornece uma opera??o direta para atribuir novas descri??es a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
